# 02 - Feature Engineering and Clustering

In the previous notebook, I inspected the raw Steam interaction data and created a user-level feature table.  
In this notebook, I use those user-level behavioral features for clustering and PCA visualization.

## 1. Import libraries and project paths

I use pandas and matplotlib for data analysis and visualization.  
For clustering, I use StandardScaler, KMeans, silhouette score, and PCA from scikit-learn.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR

## 2. Load the processed user-level data

The raw interaction data was already converted into user-level features in the previous step.  
This processed table is more suitable for clustering because each row now represents one user.

In [ ]:
user_features_path = PROCESSED_DATA_DIR / "user_features.csv"

user_features = pd.read_csv(user_features_path)
user_features.head()

In [ ]:
user_features.shape

In [ ]:
user_features.describe()

## 3. Select behavioral features for clustering

The `user_id` column is only an identifier, so I do not use it as an input feature.  
The remaining columns describe gameplay intensity, game library diversity, and purchasing behavior.

In [ ]:
feature_columns = [
    "total_hours",
    "avg_hours",
    "max_hours",
    "unique_games",
    "purchase_count",
    "total_interactions",
    "hours_per_game",
    "purchase_ratio"
]

X = user_features[feature_columns]

X.head()

## 4. Scale the features

K-Means is distance-based, so feature scaling is important.  
Without scaling, variables with larger numerical ranges could dominate the clustering result.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]

## 5. Elbow Method

I first calculate inertia values for different numbers of clusters.  
The goal is to see where adding more clusters starts to provide less improvement.

In [ ]:
k_values = range(2, 11)
inertia_values = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

inertia_values

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(list(k_values), inertia_values, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.grid(True)
plt.show()

## 6. Silhouette Score

I also calculate the silhouette score for different k values.  
Higher silhouette scores generally indicate better separated clusters.

In [ ]:
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(score)

silhouette_scores

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(list(k_values), silhouette_scores, marker="o")
plt.title("Silhouette Scores for K-Means")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()

## 7. Fit the final K-Means model

Based on the earlier project idea and the need for interpretable user profiles, I start with 4 clusters.  
This gives enough detail to separate different user behaviors without making the segments too difficult to explain.

In [ ]:
final_k = 4

kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

clustered_users = user_features.copy()
clustered_users["cluster"] = cluster_labels

clustered_users.head()

## 8. Cluster size distribution

Before interpreting the clusters, I check how many users are assigned to each cluster.

In [ ]:
clustered_users["cluster"].value_counts().sort_index()

In [ ]:
clustered_users["cluster"].value_counts().sort_index().plot(
    kind="bar",
    title="Number of Users in Each Cluster",
    xlabel="Cluster",
    ylabel="Number of Users",
    grid=True
)

plt.show()

## 9. Cluster profile summary

To understand the meaning of each cluster, I calculate the average behavioral features within each group.

In [ ]:
cluster_summary = clustered_users.groupby("cluster")[feature_columns].mean()
cluster_summary

In [ ]:
cluster_summary[["total_hours", "unique_games", "purchase_count", "hours_per_game", "purchase_ratio"]]

## 10. Simple cluster interpretation

At this stage, the clusters are still numeric labels.  
I use their average behavior to describe them in a more meaningful way.

In [ ]:
cluster_summary[["total_hours", "unique_games", "purchase_count", "hours_per_game", "purchase_ratio"]].round(2)

The exact cluster names may be updated later, but the main idea is to compare users based on:

- how much they play
- how many different games they interact with
- whether they mostly purchase or actually play
- whether their playtime is concentrated on a few games or spread across many games

## 11. PCA visualization

Since the clustering uses several features, I use PCA to reduce the data to two dimensions for visualization.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["cluster"] = cluster_labels

pca_df.head()

In [ ]:
pca.explained_variance_ratio_

In [ ]:
plt.figure(figsize=(8, 6))

for cluster_id in sorted(pca_df["cluster"].unique()):
    temp = pca_df[pca_df["cluster"] == cluster_id]
    plt.scatter(
        temp["PC1"],
        temp["PC2"],
        label=f"Cluster {cluster_id}",
        alpha=0.6,
        s=20
    )

plt.title("PCA Visualization of Steam User Clusters")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.grid(True)
plt.show()

## 12. Hierarchical clustering on a small sample

Hierarchical clustering can be useful to compare whether the user grouping structure is similar to K-Means.  
Since the full dataset has many users, I use a small sample only for a quick exploratory comparison.

In [ ]:
sample_size = 500

sample_indices = np.random.default_rng(42).choice(
    X_scaled.shape[0],
    size=sample_size,
    replace=False
)

X_sample = X_scaled[sample_indices]

agg = AgglomerativeClustering(n_clusters=4)
agg_labels = agg.fit_predict(X_sample)

pd.Series(agg_labels).value_counts().sort_index()

## 13. Save clustered users

The next notebook will use the cluster labels as a target-like variable for model comparison and interpretation.  
This does not mean the clusters are true external labels; they are labels discovered by K-Means.

In [ ]:
clustered_users_path = PROCESSED_DATA_DIR / "clustered_users.csv"

clustered_users.to_csv(clustered_users_path, index=False)

clustered_users_path

## 14. Main observations from clustering

At this stage:

- The raw interaction data has been converted into user-level behavioral features.
- Scaling was applied before K-Means because clustering is distance-based.
- Elbow Method and Silhouette Score were used to inspect possible cluster counts.
- K-Means was used to create user segments.
- PCA helped visualize the clusters in two dimensions.
- The clustered user table was saved for the modeling notebook.

The next step is to use the cluster labels for predictive modeling and interpretation.